In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
train_data=pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
print(train_data.head())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


In [3]:
train_df=pd.DataFrame(train_data)
print(train_df.columns)
print(train_df.shape)

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')
(891, 12)


## Checking if Missing Values are present or not

In [4]:
for idx in train_df.columns:
    print(f"{idx}:{sum(train_df[idx].isnull())}")
for idx in train_df.columns:
    print(f"{idx}:{train_df[idx].dtype}")

PassengerId:0
Survived:0
Pclass:0
Name:0
Sex:0
Age:177
SibSp:0
Parch:0
Ticket:0
Fare:0
Cabin:687
Embarked:2
PassengerId:int64
Survived:int64
Pclass:int64
Name:object
Sex:object
Age:float64
SibSp:int64
Parch:int64
Ticket:object
Fare:float64
Cabin:object
Embarked:object


In [5]:
Y=train_df['Survived'] 
# We drop 'Survived' and columns that act as unique identifiers (PassengerId, Name)
X=train_df.drop(['Survived','PassengerId','Name'],axis=1)

print(f"Features shape:{X.shape}")
print(f"Target shape:{Y.shape}")


Features shape:(891, 9)
Target shape:(891,)


## We will use KNNImputer to fill missing values after tuning its hyperparameters i.e no. of neighbors

In [6]:
import optuna
from sklearn.model_selection import StratifiedKFold,cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestClassifier
from category_encoders import TargetEncoder

cat_cols=['Sex','Ticket','Cabin','Embarked']
num_cols=['Pclass','Age','SibSp','Parch','Fare']

def objective(trial):
    #suggest hyperparameter for knn
    n_nb=trial.suggest_int('knn_n_neighbors',2,20) #telling optuna to pick a 

    pipeline=Pipeline([
        ('encoder',TargetEncoder(cols=cat_cols)),
        ('scaler',StandardScaler()),
        ('knn',KNNImputer(n_neighbors=n_nb)),
        ('clf',RandomForestClassifier(random_state=42))
    ])

    cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
    scores=cross_val_score(pipeline,X,Y,cv=cv,scoring='accuracy',n_jobs=-1)

    return scores.mean()

#This is the master object that records all your trials and remembers which parameters worked best.
#from direction='maximize',You are telling Optuna that a higher number is better. 
#Since your objective function returns Accuracy, you want to reach the highest percentage possible.
study=optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30) #Optuna will pass a trial into objective function, 
#get the accuracy score back, and use that information to pick a smarter number for the next trial.

print(f"Best Accuracy:{study.best_value:.4f}")
print(f"Best n_neighbors for KNNImputer:{study.best_params['knn_n_neighbors']}")

[I 2026-04-12 15:45:59,397] A new study created in memory with name: no-name-57af1ec1-aedc-49d0-8e10-36606cf6a52c
[I 2026-04-12 15:46:01,756] Trial 0 finished with value: 0.8047329106772958 and parameters: {'knn_n_neighbors': 9}. Best is trial 0 with value: 0.8047329106772958.
[I 2026-04-12 15:46:02,204] Trial 1 finished with value: 0.808097420124286 and parameters: {'knn_n_neighbors': 10}. Best is trial 1 with value: 0.808097420124286.
[I 2026-04-12 15:46:02,631] Trial 2 finished with value: 0.8047266336074321 and parameters: {'knn_n_neighbors': 13}. Best is trial 1 with value: 0.808097420124286.
[I 2026-04-12 15:46:03,066] Trial 3 finished with value: 0.8092210156299039 and parameters: {'knn_n_neighbors': 17}. Best is trial 3 with value: 0.8092210156299039.
[I 2026-04-12 15:46:03,495] Trial 4 finished with value: 0.8047266336074321 and parameters: {'knn_n_neighbors': 13}. Best is trial 3 with value: 0.8092210156299039.
[I 2026-04-12 15:46:03,922] Trial 5 finished with value: 0.804726

Best Accuracy:0.8092
Best n_neighbors for KNNImputer:17


## Using the best value obtained for no. of neighbors in our final pipeline

In [7]:
final_pipeline = Pipeline([
    ('encoder', TargetEncoder(cols=['Sex','Ticket','Cabin','Embarked'])),
    ('scaler', StandardScaler()), 
    ('knn', KNNImputer(n_neighbors=17)), 
    ('clf', RandomForestClassifier(random_state=42))
])

final_pipeline.fit(X, Y)

Pipeline(steps=[('encoder',
                 TargetEncoder(cols=['Sex', 'Ticket', 'Cabin', 'Embarked'])),
                ('scaler', StandardScaler()),
                ('knn', KNNImputer(n_neighbors=17)),
                ('clf', RandomForestClassifier(random_state=42))])

In [8]:
test_data=pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

In [9]:
test_df=pd.DataFrame(test_data)

In [10]:
X_test=test_df.drop(['PassengerId','Name'],axis=1)


In [11]:
predictions = final_pipeline.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': predictions
})

submission.to_csv('titanic_submission.csv', index=False)
print("Submission file created successfully!")

Submission file created successfully!


In [12]:
print(submission.shape)

(418, 2)
